<a href="https://colab.research.google.com/github/MohHaroon/XAI-based-ZSL-for-IIDS/blob/master/ESZSL_ZDBERTa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
SDN_train_test_data = pd.read_csv("/content/drive/MyDrive/IRP/SDN-Dataset/SDN_data.csv")

In [3]:
display(SDN_train_test_data)

,proto_number,Dur,Mean,Stddev,Min,Max,Pkts,Bytes,Spkts,Dpkts,...,Sum,TnBPSrcIP,TnBPDstIP,TnP_PSrcIP,TnP_PDstIP,TnP_PerProto,TnP_Per_Dport,N_IN_Conn_P_DstIP,N_IN_Conn_P_SrcIP,Attack
0,6.0,4.0,250.923077,471.889193,0.0,1079.0,194673.0,92007523.0,2.0,4.0,...,3262.0,1.150094e+07,1.314393e+07,24334.125000,27810.428571,97336.5,21630.333333,1.857143,1.857143,0.0
1,6.0,0.0,13.093915,13.406746,0.0,60.0,7843.0,5713792.0,0.0,0.0,...,9899.0,7.731789e+03,9.522987e+05,10.612991,1307.166667,3921.5,560.214286,126.000000,126.000000,0.0
2,6.0,638.0,45.450704,125.392502,3.0,638.0,137392.0,96000104.0,37681.0,50237.0,...,3227.0,1.066668e+07,1.200001e+07,15265.777778,17174.000000,68696.0,4906.857143,8.875000,8.875000,0.0
3,6.0,9.0,580.888889,857.833673,8.0,1725.0,371297.0,259676578.0,5.0,3.0,...,5228.0,4.327943e+07,5.193532e+07,61882.833333,74259.400000,185648.5,53042.428571,1.800000,1.800000,0.0
4,6.0,1186.0,326.818182,552.040002,0.0,1187.0,255083.0,185764615.0,93286.0,69967.0,...,3595.0,2.653780e+07,3.096077e+07,36440.428571,42513.833333,127541.5,31885.375000,1.833333,1.833333,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209995,6.0,10.0,96.487179,319.330869,0.0,1189.0,162070.0,127982168.0,15.0,17.0,...,3763.0,1.599777e+07,1.828317e+07,20258.750000,23152.857143,81035.0,7046.521739,5.571429,5.571429,1.0
209996,6.0,0.0,1.436364,2.522224,0.0,9.0,1361.0,911280.0,6.0,4.0,...,79.0,1.301829e+05,1.518800e+05,194.428571,226.833333,680.5,43.903226,9.166667,9.166667,1.0
209997,6.0,3.0,79.844444,287.184448,0.0,1143.0,155529.0,122555630.0,3.0,7.0,...,3593.0,1.750795e+07,2.042594e+07,22218.428571,25921.500000,77764.5,5981.884615,7.500000,7.500000,1.0
209998,6.0,0.0,24.028571,114.177953,0.0,687.0,93490.0,73529380.0,6.0,4.0,...,2523.0,9.191172e+06,1.050420e+07,11686.250000,13355.714286,46745.0,1669.464286,15.000000,15.000000,1.0


In [3]:
def scale_data(data):
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(data)
    return scaled_data

SDN_scaled_data = scale_data(SDN_train_test_data.drop(columns=['Attack']))

def split_data(data, labels, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=test_size, random_state=random_state)
    return X_train, X_test, y_train, y_test

data_train, data_test, labels_train, labels_test = split_data(SDN_scaled_data, SDN_train_test_data['Attack'])

EZSL

In [5]:
import numpy as np

class ESZSL_Model:
    def __init__(self, lambd=0.1, gamma=0.1):
        self.lambd = lambd
        self.gamma = gamma
        self.W = None

    def fit(self, X, S, Y):
        """
        X: Training features (N x D)
        S: Attribute matrix for seen classes (L x A)
        Y: One-hot labels (N x L)
        """
        d = X.shape[1]
        a = S.shape[1]

        # Closed-form solution: W = (X'X + λI)^-1 X' Y S (S'S + γI)^-1
        part1 = np.linalg.inv(X.T @ X + self.lambd * np.eye(d))
        part2 = X.T @ Y @ S
        part3 = np.linalg.inv(S.T @ S + self.gamma * np.eye(a))

        self.W = part1 @ part2 @ part3

    def predict(self, X_test, S_unseen):
        # Scores = X * W * S_unseen'
        scores = X_test @ self.W @ S_unseen.T
        return np.argmax(scores, axis=1)

In [8]:
# 1. Prepare Y (One-hot labels for seen classes)
# The current labels_train is binary (0.0 or 1.0), but S_seen implies 5 seen classes.
# We need Y_train to have 5 columns, where each column corresponds to a class in S_seen.
# Let's assume 0.0 maps to the first class (Normal) and 1.0 maps to the second class (DoS) in S_seen.
# The other classes (DDoS, Port Scan, Fuzzing) are assumed not present in the training labels.
import numpy as np

num_samples = labels_train.shape[0]
num_seen_classes = S_seen.shape[0] # This is 5

Y_train = np.zeros((num_samples, num_seen_classes))

# Map 0.0 to the first column (index 0) and 1.0 to the second column (index 1)
# This explicitly creates a one-hot encoding for 5 classes, where only the first two are 'active' based on labels_train.
Y_train[labels_train == 0.0, 0] = 1
Y_train[labels_train == 1.0, 1] = 1

# 2. Define your S matrices (Based on the table above)
# S_seen: attributes for Normal, DoS, DDoS, Port Scan, Fuzzing
# S_unseen: attributes for OS Fingerprinting (and others if testing GZSL)
S_seen = np.array([[0,0,0,0,1], [1,1,0,0,0], [1,1,0,0,0], [0,1,1,0,0], [0,0,0,1,0]])
S_unseen = np.array([[0,1,1,0,1]])

# 3. Train
eszsl = ESZSL_Model(lambd=0.1, gamma=0.1)
eszsl.fit(data_train, S_seen, Y_train)

# 4. Run your Stress Test Script
# Just replace 'model' with 'eszsl' in your automated_balanced_stress_test call
results_df = automated_balanced_stress_test(data_test, labels_test, eszsl, explainer, profiles)

NameError: name 'automated_balanced_stress_test' is not defined

In [9]:
# 1. Pick one sample of a known attack (e.g., DDoS)
sample_idx = 0 # Adjust to an index you know is DDoS
sample_features = data_test.iloc[sample_idx:sample_idx+1].values
true_label = labels_test.iloc[sample_idx]

# 2. Get the raw projection into Semantic Space
# Projection = X * W
semantic_projection = sample_features @ eszsl.W

# 3. Calculate similarity scores manually
# Score = Projection * S_all.T
S_all = np.vstack([S_seen, S_unseen])
all_labels = lb.classes_.tolist() + ["OS Fingerprinting"] # Assuming lb.classes_ are your seen labels

scores = semantic_projection @ S_all.T

print(f"--- Manual Test for Sample {sample_idx} ({true_label}) ---")
for i, label in enumerate(all_labels):
    print(f"Similarity with {label}: {scores[0][i]:.4f}")

AttributeError: 'numpy.ndarray' object has no attribute 'iloc'

jscjds

In [13]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelBinarizer

# Define the classes in the order they appear in your dataset
# Seen Classes: Normal, DoS, DDoS, Port Scan, Fuzzing
# Unseen Class: OS Fingerprinting (Zero-Day proxy)
class_names = ['Normal', 'DoS', 'DDoS', 'Port Scan', 'Fuzzing', 'OS Fingerprinting']

# Attributes: [Volumetric, High-Freq, Multi-Port, Payload-Heavy, Stealthy]
attributes = np.array([
    [0, 0, 0, 0, 1], # Normal
    [1, 1, 0, 0, 0], # DoS
    [1, 1, 0, 0, 0], # DDoS
    [0, 1, 1, 0, 0], # Port Scan
    [0, 0, 0, 1, 0], # Fuzzing
    [0, 1, 1, 0, 1]  # OS Fingerprinting (Unseen)
])

S_matrix = pd.DataFrame(attributes, index=class_names)

In [11]:
class ESZSL_IOT:
    def __init__(self, alpha=2, gamma=2):
        self.alpha = alpha # Feature regularization (10^alpha)
        self.gamma = gamma # Attribute regularization (10^gamma)
        self.W = None
        self.lb = LabelBinarizer()

    def fit(self, X, y, S_seen):
        """
        X: data_train (N x D)
        y: labels_train (N,)
        S_seen: Attribute matrix for seen classes (L_seen x A)
        """
        # 1. Convert labels to one-hot Ground Truth (Y)
        Y = self.lb.fit_transform(y)

        # 2. Dimensions
        d = X.shape[1]
        a = S_seen.shape[1]

        # 3. Closed-form Solution (The "One-line" solution from your reference)
        # Part 1: (XX' + λI)^-1
        part_1 = np.linalg.pinv(X.T @ X + (10**self.alpha) * np.eye(d))
        # Part 0: X' Y S
        part_0 = X.T @ Y @ S_seen
        # Part 2: (SS' + γI)^-1
        part_2 = np.linalg.pinv(S_seen.T @ S_seen + (10**self.gamma) * np.eye(a))

        self.W = part_1 @ part_0 @ part_2
        print("ESZSL Training Complete. Weight matrix W shape:", self.W.shape)

    def predict(self, X_test, S_all):
        """
        X_test: data_test
        S_all: All attributes (Seen + Unseen)
        """
        # Score = X * W * S'
        scores = X_test @ self.W @ S_all.T
        # Return indices of highest scores
        return np.argmax(scores.values, axis=1)

# Initialize and Train
# Ensure S_seen only contains classes present in labels_train
seen_classes = np.unique(labels_train)
S_seen = S_matrix.loc[seen_classes].values

model_eszsl = ESZSL_IOT(alpha=2, gamma=2)
model_eszsl.fit(data_train, labels_train, S_seen)

KeyError: "None of [Index([0.0, 1.0], dtype='float64')] are in the [index]"

In [14]:
class ESZSL_IOT:
    def __init__(self, alpha=2, gamma=2):
        self.alpha = alpha
        self.gamma = gamma
        self.W = None

    def fit(self, X, y, S_seen):
        # Convert y (0.0, 1.0) to a binary matrix Y
        # If binary (0/1), LabelBinarizer returns (N, 1).
        # For ESZSL, we need (N, L) where L is number of classes.
        from sklearn.preprocessing import OneHotEncoder
        ohe = OneHotEncoder(sparse_output=False)
        Y = ohe.fit_transform(y.values.reshape(-1, 1))

        d = X.shape[1]
        a = S_seen.shape[1]

        # Sylvester Equation Solution
        part_1 = np.linalg.pinv(X.T @ X + (10**self.alpha) * np.eye(d))
        part_0 = X.T @ Y @ S_seen
        part_2 = np.linalg.pinv(S_seen.T @ S_seen + (10**self.gamma) * np.eye(a))

        self.W = part_1 @ part_0 @ part_2

In [28]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelBinarizer

# Updated Mapping for all categories
id_to_name = {
    0.0: "Normal",
    1.0: "DoS",
    2.0: "DDoS",
    3.0: "Port Scan",
    4.0: "Fuzzing",
    5.0: "OS Fingerprinting (Zero-Day)"
}

# Update S_matrix with all 6 rows
attributes_all = np.array([
    [0, 0, 0, 0, 1], # 0.0
    [1, 1, 0, 0, 0], # 1.0
    [1, 1, 0, 0, 0], # 2.0
    [0, 1, 1, 0, 0], # 3.0
    [0, 0, 0, 1, 0], # 4.0
    [0, 1, 1, 0, 1]  # 5.0
])

S_matrix = pd.DataFrame(attributes_all, index=[0.0, 1.0, 2.0, 3.0, 4.0, 5.0])

# 2. Get the unique labels present in your training split
seen_classes = np.unique(labels_train)
print(f"Seen classes in training: {seen_classes}")

# 3. Slice S_seen safely
S_seen = S_matrix.loc[seen_classes].values

# 4. Initialize and Train
model_eszsl = ESZSL_IOT(alpha=2, gamma=2)
model_eszsl.fit(data_train, labels_train, S_seen)

Seen classes in training: [0. 1.]


In [29]:
# 1. Pick one sample from the test set (which belongs to a seen class)
sample_idx = 0
sample_features = data_test[sample_idx:sample_idx+1]
true_label_numeric = labels_test.iloc[sample_idx]

# Map true_label_numeric to its string representation using id_to_name
true_label_name = id_to_name.get(true_label_numeric, f"Unknown (numeric: {true_label_numeric})")

# 2. Get the raw projection into Semantic Space
# Projection = X * W
predicted_attributes = sample_features @ model_eszsl.W

# 3. Calculate similarity scores with all classes (seen and unseen)
S_all = S_matrix.values # Uses attributes for all defined classes
all_class_ids = S_matrix.index.tolist() # [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]

similarities = predicted_attributes @ S_all.T

print(f"--- Sanity Check for Sample {sample_idx} (True Label: {true_label_name}) ---")
for i, class_id in enumerate(all_class_ids):
    display_class_name = id_to_name.get(class_id, f"Unknown (numeric: {class_id})")
    print(f"Similarity with Class {display_class_name}: {similarities[0][i]:.4f}")

--- Sanity Check for Sample 0 (True Label: DoS) ---
Similarity with Class Normal: 0.0023
Similarity with Class DoS: 0.0116
Similarity with Class DDoS: 0.0058
Similarity with Class Port Scan: 0.0058
Similarity with Class Fuzzing: 0.0000
Similarity with Class OS Fingerprinting (Zero-Day): 0.0081
